# 第七章：希腊字母 — 期权的"仪表盘" / Chapter 7: The Greeks — An Option's Dashboard

## 你将学到 / What You Will Learn
- 什么是"希腊字母"（不是真的要学希腊语！）/ What "the Greeks" are (no, you do not have to learn Greek!)
- 每个字母代表什么风险 / Which risk each letter measures
- 怎么通过图表直观理解它们 / How to read them visually

---

## 7.1 为什么需要希腊字母？ / 7.1 Why Do We Need the Greeks?

上一章我们学会了给期权定价。但还有一个问题：

The previous chapter taught us to *price* an option. But one question remains:

> "如果市场变了，我的期权会怎样？"
>
> "If the market moves, what happens to my option?"

比如：

For example:

- 股价涨了 1块 → 我的期权价格变了多少？ → 这就是 **Delta**
  Spot up by 1 → by how much does my option price change? → **Delta**
- 股价波动变大了 → 我的期权变贵还是变便宜？ → 这就是 **Vega**
  Volatility expands → does my option get pricier or cheaper? → **Vega**
- 又过了一天 → 我的期权贬值了多少？ → 这就是 **Theta**
  One more day passes → how much value did my option lose? → **Theta**

**希腊字母就是期权的"仪表盘"**，每个字母是一个表盘，告诉你不同方向的风险。

**The Greeks are the option's dashboard** — each letter is a separate gauge telling you the risk along a different axis.

## 7.2 五个希腊字母一览 / 7.2 The Five Greeks at a Glance

| 字母 / Letter | 读法 / Pronunciation | 衡量什么？ / What It Measures | 通俗理解 / Plain-English Reading |
|------|------|-----------|---------|
| **Δ Delta** | "戴尔塔" / "del-tuh" | 标的价变1块 → 期权价变多少 / Change in option price for a 1-unit move in the underlying | 方向盘：你对涨跌多敏感 / Steering wheel — how much you tilt with the market |
| **Γ Gamma** | "伽马" / "gam-uh" | Delta 本身变化有多快 / How fast Delta itself changes | 加速度：弯道的弯曲程度 / Acceleration — how sharp the curve is |
| **Θ Theta** | "西塔" / "thay-tuh" | 每过一天 → 期权贬值多少 / Daily erosion of option value | 倒计时：时间在吃掉你的期权 / A countdown — time is eating your option |
| **ν Vega** | "维加" / "vay-guh" | 波动率变1% → 期权价变多少 / Change in option price for a 1-point move in vol | 恐慌指数：市场越动荡期权越贵 / A fear gauge — more turbulence, pricier options |
| **ρ Rho** | "柔" / "row" | 利率变1% → 期权价变多少 / Change in option price for a 1-point move in rates | 通常影响较小，了解即可 / Usually small — nice to know |

## 7.3 更直观的理解 / 7.3 An Even More Intuitive Picture

想象你买了一张**彩票**（买方看涨期权）：

Imagine you bought a **lottery ticket** (a long call option):

- **Delta** = 你中奖的概率有多大（0=不可能, 1=稳中）/ The probability of winning (0 = no chance, 1 = certain)
- **Gamma** = 概率变化有多快（快开奖时变化最剧烈）/ How fast that probability changes (most dramatic near the draw)
- **Theta** = 彩票每天都在贬值（越接近开奖，时间价值越少）/ The ticket bleeds value every day (closer to the draw, less time value)
- **Vega** = 如果彩票的号码范围扩大了（波动更大），你中奖可能性变了多少 / If the draw range widens (more volatility), how your win odds change

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 7.4 互动实验：四个仪表盘 / 7.4 Interactive Experiment: Four Dashboards

> 拖动下面的滑块，观察四张图同时变化。
>
> Drag the sliders below and watch all four charts update together.
>
> 特别注意：
> - Delta 在行权价附近变化最快
> - Gamma 在行权价附近最大（"ATM处最敏感"）
> - Theta 永远是负数（时间在吃你的钱！）
> - Vega 也在行权价附近最大
>
> Pay attention to:
> - Delta changes fastest around the strike
> - Gamma peaks at the strike ("most sensitive at-the-money")
> - Theta is always negative — time is eating your money
> - Vega is also maximized near the strike

In [ ]:
def greeks_chart(K=100, T=1.0, r_pct=5.0, sigma_pct=20.0):
    plt.close('all')
    r = r_pct / 100
    sigma = sigma_pct / 100
    S = np.linspace(50, 150, 200)

    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    delta_call = norm.cdf(d1)
    delta_put = delta_call - 1
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    theta_call = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
                  - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # Delta
    ax = axes[0][0]
    ax.plot(S, delta_call, color=C_GREEN, lw=2.5, label='Call Delta')
    ax.plot(S, delta_put, color=C_RED, lw=2.5, label='Put Delta')
    ax.axhline(y=0.5, color=C_ORANGE, ls=':', alpha=0.5, label='Delta=0.5 (ATM)')
    ax.axhline(y=0, color='gray', ls='--', alpha=0.3)
    ax.axvline(x=K, color=C_PURPLE, ls='--', alpha=0.3)
    ax.set_xlabel('Spot S')
    ax.set_ylabel('Delta')
    ax.set_title('Delta - direction sensitivity', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.annotate('Deep ITM\n(Delta -> 1)', xy=(140, delta_call[np.argmin(np.abs(S - 140))]),
                fontsize=8, color=C_GREEN, ha='center')
    ax.annotate('Deep OTM\n(Delta -> 0)', xy=(60, delta_call[np.argmin(np.abs(S - 60))]),
                fontsize=8, color='gray', ha='center')

    # Gamma
    ax = axes[0][1]
    ax.plot(S, gamma, color=C_PURPLE, lw=2.5)
    ax.fill_between(S, gamma, 0, color=C_PURPLE, alpha=0.12)
    ax.axvline(x=K, color=C_PURPLE, ls='--', alpha=0.3)
    ax.set_xlabel('Spot S')
    ax.set_ylabel('Gamma')
    ax.set_title('Gamma - rate of Delta change', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.annotate('Max near strike\n(most sensitive)', xy=(K, gamma.max()),
                xytext=(K + 18, gamma.max() * 0.7), fontsize=9, fontweight='bold', color=C_PURPLE,
                arrowprops=dict(arrowstyle='->', color=C_PURPLE, lw=1.5))

    # Theta
    ax = axes[1][0]
    ax.plot(S, theta_call, color=C_ORANGE, lw=2.5, label='Call Theta')
    ax.fill_between(S, theta_call, 0, color=C_ORANGE, alpha=0.12)
    ax.axvline(x=K, color=C_PURPLE, ls='--', alpha=0.3)
    ax.axhline(y=0, color='gray', ls='--', alpha=0.3)
    ax.set_xlabel('Spot S')
    ax.set_ylabel('Theta (per day)')
    ax.set_title('Theta - daily time decay', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.annotate('Always negative!\nTime hurts long options', xy=(K, theta_call[len(S) // 2]),
                xytext=(K + 20, theta_call.min() * 0.5), fontsize=9, color=C_ORANGE, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C_ORANGE))

    # Vega
    ax = axes[1][1]
    ax.plot(S, vega, color=C_BLUE, lw=2.5)
    ax.fill_between(S, vega, 0, color=C_BLUE, alpha=0.12)
    ax.axvline(x=K, color=C_PURPLE, ls='--', alpha=0.3)
    ax.set_xlabel('Spot S')
    ax.set_ylabel('Vega')
    ax.set_title('Vega - vol sensitivity (per 1%)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.annotate('Max near strike too', xy=(K, vega.max()),
                xytext=(K + 18, vega.max() * 0.7), fontsize=9, color=C_BLUE, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.5))

    plt.suptitle(f'Params: K={K}, T={T}y, r={r_pct}%, sigma={sigma_pct}%', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    atm = np.argmin(np.abs(S - K))
    print(f'  At ATM (S=K={K}):')
    print(f'    Delta = {delta_call[atm]:.3f}')
    print(f'    Gamma = {gamma[atm]:.5f}')
    print(f'    Theta = {theta_call[atm]:.4f} (per day)')
    print(f'    Vega  = {vega[atm]:.4f} (per 1% vol)')

interact(greeks_chart,
         K=FloatSlider(value=100, min=60, max=150, step=1, description='Strike K:'),
         T=FloatSlider(value=1.0, min=0.05, max=3.0, step=0.05, description='T (yrs):'),
         r_pct=FloatSlider(value=5.0, min=0, max=15, step=0.5, description='Rate r(%):'),
         sigma_pct=FloatSlider(value=20.0, min=5, max=60, step=1, description='Vol(%):'));

## 7.5 一张表记住所有 / 7.5 One Table to Remember Them All

| 希腊字母 / Greek | 买方(Long)视角 / Long-Position Sign | 一句话 / One-Liner |
|---------|---------------|--------|
| Delta | 正(Call)/负(Put) / Positive (call) / negative (put) | 涨跌方向的敏感度 / Sensitivity to the direction of the underlying |
| Gamma | 总是正的 / Always positive | Delta变化越快，行权价附近最大 / Measures how fast Delta changes; peaks at the strike |
| Theta | **总是负的** / **Always negative** | 时间每天都在侵蚀你的期权价值 / Time erodes your option's value every day |
| Vega | 总是正的 / Always positive | 市场波动越大，你的期权越值钱 / The more turbulent the market, the pricier your option |

> **记忆口诀**: "Delta看方向，Gamma看加速，Theta吃时间，Vega爱波动"
>
> **Mnemonic**: "Delta = direction, Gamma = acceleration, Theta = time eater, Vega = loves volatility."
>
> **下一章：波动率微笑** —— 为什么现实世界不完美？
>
> **Next chapter: the volatility smile** — why real markets never look as smooth as the textbook.